# 서울 ASOS 시간별 기온 예측 연습

**목표:** 전날 00–23시 관측값으로 오늘 00–23시 기온 24개를 한 번에 예측합니다. 학습 연습용이며 팀 서비스 모델은 아닙니다.

**실제 CSV 확인 (2026-10-13):** 서울 지점 108, 2020–2025년 원본 6개 52,602행. 지점·시각 중복 0건, 기온 빈칸 0건입니다. 매년 1월 1일 00시가 없어 전체 6시간이 빠졌습니다. 파일명 끝의 `2020`과 `2021`은 실제 데이터 연도가 서로 바뀌어 있으므로 파일명이 아닌 `일시`를 사용합니다. 원본 인코딩은 CP949, 병합본은 UTF-8 BOM입니다.

**입력:** `ASOS_서울108_2020-2025_병합.csv` 한 개 **또는** 원본 6개를 업로드하세요. 둘을 함께 올리면 중복 오류가 납니다. Colab 기본 패키지(pandas, numpy, torch, matplotlib)를 사용하며 별도 설치 명령은 없습니다. 각 셀을 위에서 아래로 실행하세요.

## 1. 준비
기본 예측 대상은 기온입니다. 습도를 연습하려면 아래 `TARGET` 한 줄만 바꿉니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TARGET = "기온(°C)"  # 습도 연습: "습도(%)"
FEATURE_COLS = ["기온(°C)", "습도(%)", "풍속(m/s)", "강수량(mm)"]
assert TARGET in FEATURE_COLS
np.random.seed(42)


## 2. CSV 선택
Colab에서는 업로드 창이 뜹니다. Drive를 쓴다면 `DRIVE_CSV_FILES`에 파일 경로를 넣으세요. 로컬 실행에서는 옆 폴더의 병합본을 자동으로 찾습니다.

In [ ]:
DRIVE_CSV_FILES = []  # 예: ["/content/drive/MyDrive/ASOS_서울108_2020-2025_병합.csv"]

if DRIVE_CSV_FILES:
    csv_paths = [Path(p) for p in DRIVE_CSV_FILES]
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_paths = [Path(name) for name in uploaded if name.lower().endswith(".csv")]
    except ImportError:
        local_dir = Path("../ASOS_데이터")
        csv_paths = sorted(local_dir.glob("ASOS_*_병합.csv"))
        if not csv_paths:
            csv_paths = sorted(local_dir.glob("OBS_ASOS_TIM_*.csv"))

if not csv_paths or any(not p.is_file() for p in csv_paths):
    raise FileNotFoundError("CSV를 업로드하거나 DRIVE_CSV_FILES 경로를 확인하세요.")
print("선택한 파일:", ", ".join(p.name for p in csv_paths))


## 3. 읽기·병합·검사
CP949 원본과 UTF-8 병합본을 모두 읽습니다. 실제 `일시`로 정렬하고 지점 108만 남깁니다. 중복이나 잘못된 시각은 멈춰서 알려줍니다.

In [ ]:
def read_asos(path):
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype={"지점": "string"})
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"인코딩을 읽을 수 없습니다: {path}")

parts = [read_asos(path) for path in csv_paths]
required = {"지점", "일시", *FEATURE_COLS}
for path, part in zip(csv_paths, parts):
    missing_cols = required - set(part.columns)
    if missing_cols:
        raise ValueError(f"{path.name}: 없는 열 {sorted(missing_cols)}")

raw = pd.concat(parts, ignore_index=True)
station_counts = raw["지점"].value_counts(dropna=False).to_dict()
raw = raw.loc[raw["지점"] == "108"].copy()
if raw.empty:
    raise ValueError("서울 ASOS 지점 108 자료가 없습니다.")
raw["일시"] = pd.to_datetime(raw["일시"], errors="coerce")
if raw["일시"].isna().any():
    raise ValueError(f"읽을 수 없는 일시 {raw['일시'].isna().sum()}개")
duplicates = int(raw["일시"].duplicated().sum())
if duplicates:
    raise ValueError(f"중복 일시 {duplicates}개. 병합본과 원본을 함께 올렸는지 확인하세요.")

raw = raw.sort_values("일시").set_index("일시")
for col in FEATURE_COLS:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
# 마지막 날이 몇 시간만 있어도(예: 2026-01-01 00시 1줄) 하루 24칸을 다 만들어 둔다 → 빈 날은 표본에서 자동 제외
hourly_index = pd.date_range(raw.index.min().normalize(), raw.index.max().normalize() + pd.Timedelta(hours=23), freq="h")
hourly = raw.reindex(hourly_index)
print("원본 지점별 행:", station_counts)
print("서울 108 행:", len(raw), "| 시간 범위:", raw.index.min(), "~", raw.index.max())
print("연도별 행:", raw.groupby(raw.index.year).size().to_dict())
print("누락 시각:", len(hourly_index) - len(raw))
print("기온 빈칸(원본 행):", int(raw["기온(°C)"].isna().sum()))
print("풍속 빈칸(원본 행):", int(raw["풍속(m/s)"].isna().sum()))


## 4. 기온 살펴보기
예시로 2025년 1월을 그립니다. 빠진 시각은 선이 끊깁니다.

In [ ]:
month = hourly.loc["2025-01", "기온(°C)"]
ax = month.plot(figsize=(11, 3), title="Seoul ASOS temperature · Jan 2025")
ax.set(xlabel="Date", ylabel="Temperature (°C)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. 입력 특징과 하루 단위 표본
입력은 **전날**의 기온·습도·풍속·강수량과 시간/연중 날짜의 사인·코사인입니다. 강수량 빈칸만 0으로 둡니다. 다른 입력의 연속 빈칸 1–2시간은 그 입력 날짜 안에서만 채웁니다. **오늘의 목표 기온은 절대 채우지 않고**, 24시간 중 하나라도 비면 그날 표본을 제외합니다.

In [ ]:
def fill_short_gaps_within_day(series):
    def fill_day(day_values):
        missing = day_values.isna()
        runs = missing.ne(missing.shift()).cumsum()
        short_gap = missing & missing.groupby(runs).transform("sum").le(2)
        candidate = day_values.interpolate(limit_direction="both")
        return day_values.where(~short_gap, candidate)
    return series.groupby(series.index.normalize(), group_keys=False).apply(fill_day)

inputs = hourly[FEATURE_COLS].copy()
inputs["강수량(mm)"] = inputs["강수량(mm)"].fillna(0)
for col in ("기온(°C)", "습도(%)", "풍속(m/s)"):
    inputs[col] = fill_short_gaps_within_day(inputs[col])

hours = hourly_index.hour.to_numpy()
day_of_year = hourly_index.dayofyear.to_numpy()
inputs["시각_sin"] = np.sin(2 * np.pi * hours / 24)
inputs["시각_cos"] = np.cos(2 * np.pi * hours / 24)
inputs["연중일_sin"] = np.sin(2 * np.pi * day_of_year / 366)
inputs["연중일_cos"] = np.cos(2 * np.pi * day_of_year / 366)
feature_names = list(inputs.columns)

samples = []
days = pd.date_range(hourly_index[0].normalize() + pd.Timedelta(days=1),
                     hourly_index[-1].normalize(), freq="D")
excluded = {"target_missing": 0, "input_missing": 0, "previous_23_missing": 0}
for day in days:
    prev_hours = pd.date_range(day - pd.Timedelta(days=1), periods=24, freq="h")
    today_hours = pd.date_range(day, periods=24, freq="h")
    x_day = inputs.loc[prev_hours].to_numpy(dtype=float)
    y_day = hourly.loc[today_hours, TARGET].to_numpy(dtype=float)  # 원본 목표값
    last_observed = hourly.at[prev_hours[-1], TARGET]
    if np.isnan(y_day).any():
        excluded["target_missing"] += 1
    elif np.isnan(x_day).any():
        excluded["input_missing"] += 1
    elif pd.isna(last_observed):
        excluded["previous_23_missing"] += 1
    else:
        samples.append((day, x_day, y_day, float(last_observed)))

sample_dates = pd.DatetimeIndex([item[0] for item in samples])
X = np.stack([item[1] for item in samples]).astype("float32")
y = np.stack([item[2] for item in samples]).astype("float32")
last_temp = np.array([item[3] for item in samples], dtype="float32")
y_delta = y - last_temp[:, None]
print("표본:", len(samples), "| X:", X.shape, "| y:", y.shape)
print("제외한 날:", excluded)


## 6. 시간순 분리·표준화·기준 예측
마지막 해를 시험으로 떼어 둡니다. 나머지의 마지막 10%는 조기 종료를 위한 검증용입니다. 입력 평균·표준편차는 학습 부분에서만 구합니다. 기준 예측은 ‘오늘 = 어제 같은 시각’입니다.

In [ ]:
test_year = int(sample_dates.year.max())
trainval_idx = np.flatnonzero(sample_dates.year < test_year)
test_idx = np.flatnonzero(sample_dates.year == test_year)
if len(trainval_idx) < 20 or len(test_idx) == 0:
    raise ValueError("시간 분리에 필요한 연도별 표본이 부족합니다.")
val_count = max(1, int(np.ceil(len(trainval_idx) * 0.1)))
train_idx = trainval_idx[:-val_count]
val_idx = trainval_idx[-val_count:]

mean = X[train_idx].mean(axis=(0, 1), keepdims=True)
std = X[train_idx].std(axis=(0, 1), keepdims=True)
std = np.where(std < 1e-6, 1, std)
X_scaled = ((X - mean) / std).astype("float32")

baseline = X[test_idx, :, feature_names.index(TARGET)]
baseline_mae = float(np.mean(np.abs(baseline - y[test_idx])))
print("학습:", len(train_idx), "| 검증:", len(val_idx), "| 시험:", len(test_idx))
print("시험 연도:", test_year, "| 어제 그대로 MAE:", round(baseline_mae, 3), "°C" if TARGET == "기온(°C)" else "%")


## 7. 작은 MLP 학습
CPU로도 연습할 수 있는 크기입니다. 100회 안에서 검증 오차가 12회 연속 좋아지지 않으면 멈춥니다. 시험 자료는 학습·조기 종료에 쓰지 않습니다.

In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
model = nn.Sequential(
    nn.Flatten(), nn.Linear(24 * X.shape[2], 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 24)
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(torch.from_numpy(X_scaled[train_idx]),
                                  torch.from_numpy(y_delta[train_idx])),
                    batch_size=64, shuffle=True)
val_x = torch.from_numpy(X_scaled[val_idx])
val_y = torch.from_numpy(y_delta[val_idx])
best_loss, best_state, patience_left = float("inf"), None, 12
for epoch in range(1, 101):
    model.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = float(loss_fn(model(val_x), val_y))
    if val_loss < best_loss - 1e-4:
        best_loss, best_state, patience_left = val_loss, copy.deepcopy(model.state_dict()), 12
    else:
        patience_left -= 1
    if epoch == 1 or epoch % 10 == 0 or patience_left == 0:
        print(f"{epoch:3d}회 | 검증 MSE {val_loss:.3f}")
    if patience_left == 0:
        break
model.load_state_dict(best_state)
print("학습 종료:", epoch, "회")


## 8. 시험 결과
MAE는 예측값과 실제값의 절대 차이 평균입니다. 기준 예측과 MLP를 같은 2025년 시험 날짜에서 비교합니다.

In [ ]:
model.eval()
with torch.no_grad():
    pred_delta = model(torch.from_numpy(X_scaled[test_idx])).numpy()
mlp_pred = pred_delta + last_temp[test_idx, None]
mlp_mae = float(np.mean(np.abs(mlp_pred - y[test_idx])))
unit = "°C" if TARGET == "기온(°C)" else "pp"
display(pd.DataFrame({"방법": ["어제 그대로", "MLP"], "시험 MAE": [baseline_mae, mlp_mae], "단위": [unit, unit]}))


## 9. 시간별 오차
0–23시 중 언제 오차가 큰지 봅니다.

In [ ]:
hours = np.arange(24)
base_hour_mae = np.mean(np.abs(baseline - y[test_idx]), axis=0)
mlp_hour_mae = np.mean(np.abs(mlp_pred - y[test_idx]), axis=0)
plt.figure(figsize=(10, 3.5))
plt.plot(hours, base_hour_mae, marker="o", label="Yesterday")
plt.plot(hours, mlp_hour_mae, marker="o", label="MLP")
plt.xticks(hours)
plt.xlabel("Hour")
plt.ylabel(f"MAE ({unit})")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## 10. 하루 비교
시험 자료의 첫날을 예로 봅니다. 이 그림 하나가 전체 성능을 대표하지는 않습니다.

In [ ]:
i = 0
plt.figure(figsize=(10, 3.5))
plt.plot(hours, y[test_idx][i], marker="o", label="Actual")
plt.plot(hours, baseline[i], marker="o", label="Yesterday")
plt.plot(hours, mlp_pred[i], marker="o", label="MLP")
plt.title(f"Test date: {sample_dates[test_idx[i]].date()}")
plt.xticks(hours)
plt.xlabel("Hour")
plt.ylabel(f"Observed value ({unit})")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


### 습도 예측으로 바꾸기
2번 셀의 `TARGET = "기온(°C)"` 한 줄만 `TARGET = "습도(%)"`로 바꾸고 위에서부터 다시 실행하세요. 기본 노트북은 기온 예측입니다.